# Anomaly Detection — Exercises

Companion to the note [Anomaly Detection](Anomaly%20Detection.md).

Practise the detectors in the note: z-score, IQR, Mahalanobis (classical and robust), Gaussian density thresholds, Isolation Forest, Local Outlier Factor, PCA reconstruction error and simple time-series detectors, and how to evaluate them under extreme class imbalance with PR-AUC. By hand first, then from scratch in NumPy, checked against scikit-learn.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×5 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import pandas as pd
from scipy.stats import zscore, multivariate_normal
from scipy.spatial.distance import mahalanobis
from sklearn.covariance import EmpiricalCovariance, MinCovDet
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
rng = np.random.default_rng(1)
# Shared 2-D data: correlated normal points plus a tight group of anomalies (label 1)
X_norm = rng.multivariate_normal([0, 0], [[1, .9], [.9, 1]], 400)
X_out = rng.multivariate_normal([3, -3], [[.2, 0], [0, .2]], 40)
X_ad = np.r_[X_norm, X_out]
y_ad = np.r_[np.zeros(400), np.ones(40)]

## Part A · Concepts

### Exercise 1 · Why PR-AUC?
*🧠 Concept · ★☆☆*

A fraud data set has 0.1% fraud. Explain why (a) accuracy and (b) ROC-AUC can look excellent for a useless or weak detector, and why the
note recommends PR-AUC (average precision). What is the PR-AUC of a random scorer? See [[Model Evaluation and Metrics]].

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

ROC uses the false positive *rate* $FP/(FP+TN)$; with a huge number of negatives, many false positives still give a small rate. Precision $TP/(TP+FP)$ does not hide them.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Predicting "normal" for everything gives 99.9% accuracy and catches no fraud.
(b) With 999 000 negatives, flagging 10 000 normal transactions is only a 1% FPR, so ROC-AUC can be 0.99 while 9 of every 10 alerts are false. Analysts care about precision: how many alerts are real.
PR-AUC focuses on the positive class. A random scorer has precision equal to the prevalence at every recall, so its PR-AUC is $\approx$ the prevalence (here 0.001), whereas its ROC-AUC is 0.5. Always compare PR-AUC to this baseline (Exercise 9).

</details>

### Exercise 2 · Outlier vs novelty detection
*🧠 Concept · ★☆☆*

scikit-learn distinguishes **outlier detection** (training data contaminated, find the outliers in it) and **novelty detection**
(training data clean, decide whether *new* points are abnormal). For each scenario say which it is and pick a method from the note:

(a) Clean sensor logs from a machine known to be healthy; monitor it from now on.  (b) A one-off audit of last year's expense claims.
(c) Images of good products on an assembly line; flag defects.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Is there a clean reference set? Do you need to score points that were not available at training time?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Novelty detection**: fit on clean data, score new data. One-class SVM ([[Support Vector Machines]]), Gaussian density, or LOF with `novelty=True`.
(b) **Outlier detection**: everything is available at once and may be contaminated. Isolation Forest or LOF (`fit_predict`), robust statistics.
(c) **Novelty detection** on images: reconstruction error of [[PCA]] or an autoencoder ([[Generative Models]]) trained on good products.
LOF exists in both modes: as outlier detector it scores training points; with `novelty=True` it scores unseen points only.

</details>

### Exercise 3 · Which detector, and why
*🧠 Concept · ★★☆*

For each method in the note, state its core assumption about "normal" data and one failure case:
z-score, Mahalanobis, Isolation Forest, LOF, PCA reconstruction error.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about: global vs local, linear vs non-linear structure, sensitivity to contamination and to feature scale.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **z-score**: each feature is unimodal and roughly Gaussian; univariate only. Fails on anomalies that are only unusual in *combination* (e.g. tall *and* light), and the mean/std are themselves pulled by outliers (Exercise 5).
- **Mahalanobis**: normal data is one elliptical (Gaussian-like) cloud. Fails on multimodal or curved data; classical estimates are masked by contamination (Exercise 11).
- **Isolation Forest**: anomalies are few and different, so random axis-aligned splits isolate them quickly. Struggles with anomalies in dense clusters (masking/swamping) and with high-dimensional noise features.
- **LOF**: compares local density to neighbours, so handles clusters of different density. Sensitive to $k$, $O(N^2)$ naively, needs meaningful distances (scale features).
- **PCA reconstruction**: normal data lies near a linear subspace. Fails if anomalies lie *inside* the subspace (unusual scores but small reconstruction error) or the structure is non-linear (use an autoencoder).

</details>

### Exercise 4 · Masking
*🧠 Concept · ★★☆*

Explain the **masking** effect: why can a cluster of outliers make each other invisible to a z-score or classical-Mahalanobis rule?
What makes a detector robust to it? (Exercise 5 shows a 1-D example.)

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The detector's own estimates (mean, standard deviation, covariance) are computed from the contaminated data.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Outliers inflate the estimated spread (std, covariance) and pull the mean towards themselves; then each outlier's standardized distance shrinks below the threshold. With several outliers together, each "hides behind" the others.
Robust detectors use estimates with a high **breakdown point**: median and MAD instead of mean and std, IQR fences, Minimum Covariance Determinant (MCD) instead of the sample covariance, or methods that do not estimate a global location at all (Isolation Forest, LOF).

</details>

## Part B · By hand

### Exercise 5 · z-score and its weakness
*✍️ By hand · ★☆☆*

Data: $10, 12, 11, 13, 50$. Compute the z-score of 50 using the population standard deviation (divide by $N$).
Is it flagged by the common rule $|z|>3$? Then compute the robust score $\frac{x-\text{median}}{1.4826\cdot\text{MAD}}$ for 50.

In [ ]:
z50 = None
robust50 = None

_x = np.array([10, 12, 11, 13, 50.])
check('z-score', z50, zscore(_x)[-1])
_med = np.median(_x); check('robust z', robust50, (50 - _med) / (1.4826 * np.median(np.abs(_x - _med))))

<details>
<summary><b>💡 Hint 1</b></summary>

Mean $=19.2$; squared deviations sum to $1190.8$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Median $=12$; absolute deviations from it are $2,0,1,1,38$, so MAD $=1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\bar x=19.2$, $\sigma=\sqrt{1190.8/5}=\sqrt{238.16}\approx15.43$, $z=30.8/15.43\approx2.00$: **not** flagged, although 50 is obviously anomalous. The outlier inflated $\sigma$. In fact, with population std no point of a sample of size $N$ can ever have $|z|>\sqrt{N-1}$ (Samuelson's inequality), here $2$, so the rule $|z|>3$ can *never* fire on 5 points.
Robust: median $12$, MAD $=1$, score $=38/1.4826\approx25.6$: clearly flagged.

```python
z50 = 30.8 / np.sqrt(238.16)
robust50 = 38 / 1.4826
```

</details>

### Exercise 6 · IQR fences
*✍️ By hand · ★☆☆*

Data: $2,4,4,5,6,7,8,30$. Using linear-interpolation quartiles (NumPy's default: position $(n-1)p$ in the sorted data),
compute $Q_1$, $Q_3$, the IQR and Tukey's upper fence $Q_3+1.5\,\mathrm{IQR}$. Which points are flagged?

In [ ]:
upper_fence = None
n_flagged = None

_x = np.array([2, 4, 4, 5, 6, 7, 8, 30.]); _q1, _q3 = np.percentile(_x, [25, 75])
check('upper fence', upper_fence, _q3 + 1.5 * (_q3 - _q1))
check('number flagged (both fences)', n_flagged, int(((_x > _q3 + 1.5 * (_q3 - _q1)) | (_x < _q1 - 1.5 * (_q3 - _q1))).sum()))

<details>
<summary><b>💡 Hint</b></summary>

$Q_1$ sits at position $7\cdot0.25=1.75$ (0-indexed): between $x_{(1)}=4$ and $x_{(2)}=4$. $Q_3$ at $5.25$: between $7$ and $8$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$Q_1=4$, $Q_3=7+0.25\cdot1=7.25$, IQR $=3.25$, upper fence $=7.25+4.875=12.125$, lower fence $=4-4.875=-0.875$.
Only **30** is flagged. Quartiles barely move when an extreme value is added, which is why IQR fences resist masking (other quartile conventions give slightly different numbers).

```python
upper_fence = 12.125
n_flagged = 1
```

</details>

### Exercise 7 · Mahalanobis vs Euclidean
*✍️ By hand · ★★☆*

Normal data has mean $0$ and covariance $\Sigma=\begin{pmatrix}1&0.8\\0.8&1\end{pmatrix}$. Compute the squared Mahalanobis distances
$d^2=x^\top\Sigma^{-1}x$ of $a=(1,1)$ and $b=(1,-1)$. Both have the same Euclidean norm; which is more anomalous?

In [ ]:
d2_a = None
d2_b = None

_Si = np.linalg.inv([[1, .8], [.8, 1]])
check('d^2(a)', d2_a, mahalanobis([1, 1], [0, 0], _Si) ** 2)
check('d^2(b)', d2_b, mahalanobis([1, -1], [0, 0], _Si) ** 2)

<details>
<summary><b>💡 Hint</b></summary>

$\Sigma^{-1}=\frac{1}{0.36}\begin{pmatrix}1&-0.8\\-0.8&1\end{pmatrix}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$d^2(a)=\frac{1+1-1.6}{0.36}=\frac{0.4}{0.36}\approx1.11$, $d^2(b)=\frac{1+1+1.6}{0.36}=10$.
$b$ goes *against* the correlation, so it is far more unusual. For Gaussian data $d^2\sim\chi^2_d$; with $d=2$ the 99% quantile is $9.21$, so $b$ would be flagged and $a$ not.

```python
d2_a = 0.4 / 0.36
d2_b = 10.0
```

</details>

### Exercise 8 · Isolation Forest score
*✍️ By hand · ★★☆*

Isolation Forest scores a point by $s(x)=2^{-\mathbb E[h(x)]/c(n)}$, where $h$ is the path length to isolate $x$ and
$c(n)=2H(n-1)-\frac{2(n-1)}n$, $H(i)\approx\ln i+0.5772$, is the average path length of an unsuccessful search in a binary search tree.
With subsample size $n=256$: compute $c(256)$ and the score for $\mathbb E[h]=4$. What score does a point with $\mathbb E[h]=c(n)$ get?

In [ ]:
c256 = None
s_h4 = None

_c = 2 * (np.log(255) + np.euler_gamma) - 2 * 255 / 256
check('c(256)', c256, _c, tol=1e-3)
check('score for E[h]=4', s_h4, 2 ** (-4 / _c), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\ln255\approx5.541$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$c(256)=2(5.541+0.577)-1.992\approx10.245$. $s=2^{-4/10.245}=2^{-0.390}\approx0.763$: a short path, so likely an anomaly (scores close to 1 are anomalous).
A point with average path length $c(n)$ gets $2^{-1}=0.5$; if all points score $\approx0.5$ the sample has no distinct anomalies.

```python
c256 = 2 * (np.log(255) + 0.5772) - 2 * 255 / 256
s_h4 = 2 ** (-4 / c256)
```

</details>

### Exercise 9 · Baselines for PR-AUC and ROC-AUC
*✍️ By hand · ★☆☆*

A test set has 10 000 points of which 50 are anomalies. What PR-AUC (average precision) and ROC-AUC do you expect from a detector
that outputs random scores? A colleague reports PR-AUC $=0.05$. Is that good?

In [ ]:
ap_random = None
roc_random = None

_y = np.r_[np.ones(50), np.zeros(9950)]; _r = np.random.default_rng(0)
_ap = np.mean([average_precision_score(_y, _r.random(10000)) for _ in range(30)])
check('PR-AUC of random scores', ap_random, _ap, tol=2e-3)
check('ROC-AUC of random scores', roc_random, 0.5, tol=1e-9)

<details>
<summary><b>💡 Hint</b></summary>

For random scores, precision at any threshold is the fraction of positives in the flagged set.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Random: PR-AUC $\approx50/10000=0.005$, ROC-AUC $=0.5$. So PR-AUC $=0.05$ is **10 times** better than chance, though still a weak detector in absolute terms (on average only about 1 in 20 alerts is real). Always report the prevalence next to PR-AUC.

```python
ap_random = 0.005
roc_random = 0.5
```

</details>

## Part C · Code

### Exercise 10 · Gaussian density threshold
*💻 Code · ★★☆*

The classic recipe: fit a Gaussian $\mathcal N(\mu,\Sigma)$ on (mostly) normal **training** data, flag $p(x)<\varepsilon$, and choose $\varepsilon$
on a labelled **validation** set by maximizing F1. Implement it with the data below: fit on `X_tr`, choose $\varepsilon$ among the validation
log-densities, and store the test F1 in `f1_test`.

In [ ]:
r2 = np.random.default_rng(5)
X_tr = r2.multivariate_normal([0, 0], [[1, .9], [.9, 1]], 1000)
def make_eval(n_norm, n_anom):
    Xa = r2.uniform(-4, 4, (n_anom, 2))
    return np.r_[r2.multivariate_normal([0, 0], [[1, .9], [.9, 1]], n_norm), Xa], np.r_[np.zeros(n_norm), np.ones(n_anom)]
X_val, y_val = make_eval(500, 20)
X_te, y_te = make_eval(500, 20)
f1_test = None

check('test F1 is high', None if f1_test is None else f1_test > 0.7, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Work with log-densities: `multivariate_normal(mu, S).logpdf(X)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Try every validation log-density as a threshold and keep the one with the best `f1_score(y_val, logp < t)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Fitting the full covariance (not per-feature variances) matters here: the uniform anomalies that sit *across* the correlation are easy to spot only with the correlation in the model.
Choosing $\varepsilon$ on validation data and reporting on separate test data avoids an optimistic F1. Some uniform anomalies fall inside the normal cloud and can never be detected, so F1 is high but below 1.

```python
r2 = np.random.default_rng(5)
X_tr = r2.multivariate_normal([0, 0], [[1, .9], [.9, 1]], 1000)
def make_eval(n_norm, n_anom):
    Xa = r2.uniform(-4, 4, (n_anom, 2))
    return np.r_[r2.multivariate_normal([0, 0], [[1, .9], [.9, 1]], n_norm), Xa], np.r_[np.zeros(n_norm), np.ones(n_anom)]
X_val, y_val = make_eval(500, 20)
X_te, y_te = make_eval(500, 20)
dens = multivariate_normal(X_tr.mean(0), np.cov(X_tr, rowvar=False))
lp_val = dens.logpdf(X_val)
best_t = max(np.unique(lp_val), key=lambda t: f1_score(y_val, lp_val < t))
f1_test = f1_score(y_te, dens.logpdf(X_te) < best_t)
print(best_t, f1_test)
```

</details>

### Exercise 11 · Classical vs robust Mahalanobis
*💻 Code · ★★☆*

On the shared contaminated data `X_ad` (labels `y_ad`, 40 clustered anomalies), compute squared Mahalanobis distances with
(a) the classical mean/covariance and (b) the **Minimum Covariance Determinant** (`MinCovDet`). Store the PR-AUCs in `ap_classic`, `ap_robust`
and plot both 97.5% $\chi^2_2$ tolerance ellipses or simply the scores.

In [ ]:
ap_classic = None
ap_robust = None

_d = X_ad - X_ad.mean(0); _ref = np.einsum('ij,jk,ik->i', _d, np.linalg.inv(np.cov(X_ad, rowvar=False, bias=True)), _d)
check('classical AP (vs own computation)', ap_classic, average_precision_score(y_ad, _ref), tol=1e-6)
check('robust estimate unmasks the cluster', None if ap_robust is None else ap_robust > ap_classic + 0.05, True)

<details>
<summary><b>💡 Hint</b></summary>

`EmpiricalCovariance().fit(X).mahalanobis(X)` returns squared distances; same API for `MinCovDet(random_state=0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The anomaly cluster sits against the correlation, so it pulls the classical covariance towards it and weakens the correlation estimate: some anomalies look less extreme and some normal points more (PR-AUC $\approx0.89$).
MCD fits the covariance on the most concentrated half of the data, which excludes the anomalies, and ranks them perfectly (PR-AUC $=1$).

```python
ap_classic = average_precision_score(y_ad, EmpiricalCovariance().fit(X_ad).mahalanobis(X_ad))
ap_robust = average_precision_score(y_ad, MinCovDet(random_state=0).fit(X_ad).mahalanobis(X_ad))
print(ap_classic, ap_robust)
```

</details>

### Exercise 12 · Reconstruction error with PCA
*💻 Code · ★★☆*

Use [[PCA]] as a novelty detector on the digits: train on the first 120 images of digit **1**, then score the remaining 1s plus 15 images of
digit **0** (anomalies) by squared reconstruction error with $k=10$ components. Store the PR-AUC in `ap_pca`.

In [ ]:
dig = load_digits()
X1 = dig.data[dig.target == 1]; X0 = dig.data[dig.target == 0][:15]
X_train_1, X_test = X1[:120], np.r_[X1[120:], X0]
y_test = np.r_[np.zeros(len(X1) - 120), np.ones(15)]
ap_pca = None

check('PCA reconstruction detects the zeros', None if ap_pca is None else ap_pca > 0.95, True)

<details>
<summary><b>💡 Hint</b></summary>

`p = PCA(10).fit(X_train_1)`; reconstruction `p.inverse_transform(p.transform(X))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The 10-dimensional subspace captures how 1s vary; a 0 cannot be expressed in it, so its reconstruction error is large and the zeros are ranked first (PR-AUC $\approx1$).
The same idea with a non-linear encoder/decoder is the autoencoder detector from [[Generative Models]].

```python
dig = load_digits()
X1 = dig.data[dig.target == 1]; X0 = dig.data[dig.target == 0][:15]
X_train_1, X_test = X1[:120], np.r_[X1[120:], X0]
y_test = np.r_[np.zeros(len(X1) - 120), np.ones(15)]
p = PCA(10).fit(X_train_1)
err = ((X_test - p.inverse_transform(p.transform(X_test))) ** 2).sum(1)
ap_pca = average_precision_score(y_test, err)
```

</details>

### Exercise 13 · Local Outlier Factor from scratch
*💻 Code · ★★★*

Implement `lof(X, k)` exactly as scikit-learn does (neighbours exclude the point itself):

- $k\text{-dist}(o)$ = distance to the $k$-th nearest neighbour; reach-dist$_k(p,o)=\max\{k\text{-dist}(o),\ d(p,o)\}$;
- $\mathrm{lrd}(p)=1/\big(\text{mean}_{o\in N_k(p)}\text{reach-dist}_k(p,o)\big)$;
- $\mathrm{LOF}(p)=\text{mean}_{o\in N_k(p)}\mathrm{lrd}(o)/\mathrm{lrd}(p)$.

Compare with `LocalOutlierFactor(n_neighbors=20).fit(X).negative_outlier_factor_` on the data below (two clusters of very different density plus a few anomalies).

In [ ]:
r3 = np.random.default_rng(3)
X_lof = np.r_[r3.normal(0, 0.3, (100, 2)), r3.normal(5, 1.5, (100, 2)), [[1.5, 1.5], [-1.5, 1.0], [5, -2.5]]]
def lof(X, k=20):
    # TODO: return the LOF score of every point (higher = more anomalous)
    return None

lof_scores = lof(X_lof)

check('matches sklearn', lof_scores, -LocalOutlierFactor(n_neighbors=20).fit(X_lof).negative_outlier_factor_, tol=1e-6)

<details>
<summary><b>💡 Hint 1</b></summary>

`NearestNeighbors(n_neighbors=k+1).fit(X).kneighbors(X)` returns the point itself first: drop column 0.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`kdist = dist[:, -1]`; `reach = np.maximum(dist, kdist[idx])`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

LOF $\approx1$ means the point is as dense as its neighbours; $\gg1$ means sparser. The point $(1.5,1.5)$ sits far from the *dense* cluster, and LOF flags it strongly, whereas a global distance-to-mean rule would rank it below normal points of the wide cluster. That is the "local" in LOF ([[k-Nearest Neighbors]]).
(sklearn adds $10^{-10}$ to the mean reach distance to avoid division by zero on duplicates.)

```python
r3 = np.random.default_rng(3)
X_lof = np.r_[r3.normal(0, 0.3, (100, 2)), r3.normal(5, 1.5, (100, 2)), [[1.5, 1.5], [-1.5, 1.0], [5, -2.5]]]
def lof(X, k=20):
    dist, idx = NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X)
    dist, idx = dist[:, 1:], idx[:, 1:]
    kdist = dist[:, -1]
    reach = np.maximum(dist, kdist[idx])
    lrd = 1.0 / (reach.mean(1) + 1e-10)
    return lrd[idx].mean(1) / lrd

lof_scores = lof(X_lof)
print(np.round(lof_scores[-3:], 2))
```

</details>

### Exercise 14 · Isolation Forest vs LOF, evaluated properly
*💻 Code · ★★☆*

On `X_ad, y_ad` compare `IsolationForest(random_state=0)` and `LocalOutlierFactor(n_neighbors=20)`. Higher must mean more anomalous, so negate
sklearn's `score_samples` / `negative_outlier_factor_`. Store PR-AUC in `ap_if`, `ap_lof` and ROC-AUC in `roc_if`, `roc_lof`.
Why does LOF struggle here? Try `n_neighbors=50` and store its PR-AUC in `ap_lof50`.

In [ ]:
ap_if = None
ap_lof = None
roc_if = None
roc_lof = None
ap_lof50 = None

_prev = y_ad.mean()
check('IF well above the random baseline', None if ap_if is None else ap_if > 3 * _prev, True)
check('larger k helps LOF a lot', None if ap_lof50 is None else ap_lof50 > ap_lof + 0.3, True)
check('Isolation Forest PR-AUC', ap_if, average_precision_score(y_ad, -IsolationForest(random_state=0).fit(X_ad).score_samples(X_ad)), tol=1e-9)

<details>
<summary><b>💡 Hint</b></summary>

Prevalence is $40/440\approx0.09$, the PR-AUC of random guessing.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The 40 anomalies form their **own dense cluster**. LOF with $k=20<40$ compares each anomaly to its anomalous neighbours, which are just as dense, so LOF $\approx1$: they look normal (*masking*), and LOF is at chance level (PR-AUC $\approx0.10$, ROC-AUC $\approx0.46$). With $k=50$ the neighbourhoods reach into the normal data and PR-AUC jumps to $\approx0.70$.
Isolation Forest does better (PR-AUC $\approx0.43$) but also suffers from masking: a dense group needs more splits to isolate than a lone point. Note its ROC-AUC of $\approx0.94$ looks excellent, while PR-AUC reveals that many normal points are ranked above anomalies. Compare with the robust Mahalanobis detector in Exercise 11, which suits this elliptical data best.

```python
s_if = -IsolationForest(random_state=0).fit(X_ad).score_samples(X_ad)
s_lof = -LocalOutlierFactor(n_neighbors=20).fit(X_ad).negative_outlier_factor_
ap_if, ap_lof = average_precision_score(y_ad, s_if), average_precision_score(y_ad, s_lof)
roc_if, roc_lof = roc_auc_score(y_ad, s_if), roc_auc_score(y_ad, s_lof)
s_lof50 = -LocalOutlierFactor(n_neighbors=50).fit(X_ad).negative_outlier_factor_
ap_lof50 = average_precision_score(y_ad, s_lof50)
print(f"IF: AP={ap_if:.3f} ROC={roc_if:.3f} | LOF20: AP={ap_lof:.3f} ROC={roc_lof:.3f} | LOF50: AP={ap_lof50:.3f}")
```

</details>

### Exercise 15 · Spikes in a time series
*💻 Code · ★★☆*

The note suggests using **residuals of a forecasting model** for time series. Use the simplest robust "forecast": a centred rolling median
(window 11). Flag points whose residual exceeds $5\times1.4826\cdot\text{MAD}$ of the residuals. Store the flagged indices in `flagged`
and compute recall and precision against the injected spike positions `spike_idx`.

In [ ]:
r4 = np.random.default_rng(1)
t = np.arange(1000)
series = np.sin(2 * np.pi * t / 50) + 0.2 * r4.normal(size=1000)
spike_idx = r4.choice(np.arange(20, 980), 10, replace=False)
series[spike_idx] += r4.choice([-1, 1], 10) * 2.0
flagged = None

check('recall of spikes', None if flagged is None else np.isin(spike_idx, flagged).mean(), 1.0)
check('precision >= 0.7', None if flagged is None else np.isin(flagged, spike_idx).mean() >= 0.7, True)

<details>
<summary><b>💡 Hint</b></summary>

`pd.Series(series).rolling(11, center=True, min_periods=1).median()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The median ignores a single spike, so the residual at a spike is about $\pm2$ while normal residuals are noise-sized. MAD is robust to the spikes themselves (a std would be inflated by them).
All 10 spikes are found with a couple of false alarms (precision $\approx0.8$). Lower thresholds raise recall on smaller spikes at the cost of precision: the usual PR trade-off.

```python
r4 = np.random.default_rng(1)
t = np.arange(1000)
series = np.sin(2 * np.pi * t / 50) + 0.2 * r4.normal(size=1000)
spike_idx = r4.choice(np.arange(20, 980), 10, replace=False)
series[spike_idx] += r4.choice([-1, 1], 10) * 2.0
resid = series - pd.Series(series).rolling(11, center=True, min_periods=1).median().values
mad = np.median(np.abs(resid - np.median(resid)))
flagged = np.flatnonzero(np.abs(resid) > 5 * 1.4826 * mad)
print(len(flagged), np.isin(spike_idx, flagged).mean(), np.isin(flagged, spike_idx).mean())
```

</details>

### Exercise 16 · Change-point detection with CUSUM
*💻 Code · ★★★*

A process has mean 0 for 300 steps, then shifts to mean 1 (unit noise). Implement the one-sided **CUSUM** detector
$g_t=\max(0,\ g_{t-1}+x_t-\mu_0-\kappa)$, alarm when $g_t>h$, with $\mu_0$ estimated from the first 100 points, $\kappa=0.5$, $h=5$.
Store the first alarm time in `alarm_t`. Also run it on 300 points without a shift and count false alarms (restart $g$ at 0 after each alarm).

In [ ]:
r5 = np.random.default_rng(1)
x_cp = np.r_[r5.normal(0, 1, 300), r5.normal(1, 1, 200)]
def cusum(x, mu0, kappa=0.5, h=5.0):
    # TODO: return a list of alarm indices (reset g to 0 after an alarm)
    return None

alarms = cusum(x_cp, x_cp[:100].mean())
alarm_t = None if alarms is None else alarms[0]

check('alarm shortly after the change at t=300', None if alarm_t is None else 300 <= alarm_t <= 330, True)

<details>
<summary><b>💡 Hint</b></summary>

$\kappa$ is usually half the shift you want to detect; $h$ trades detection delay against the false-alarm rate (average run length).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Before the change $x_t-\mu_0-\kappa$ has mean $-0.5$, so $g_t$ keeps getting pulled back to 0; after it the drift is $+0.5$ per step and $g$ crosses $h=5$ after roughly $5/0.5=10$ steps (alarm here at $t=308$).
On data without a shift you typically get no alarms in 300 steps. Larger $h$ = fewer false alarms but longer delay. A two-sided version runs a second CUSUM on $-x$.

```python
r5 = np.random.default_rng(1)
x_cp = np.r_[r5.normal(0, 1, 300), r5.normal(1, 1, 200)]
def cusum(x, mu0, kappa=0.5, h=5.0):
    g, out = 0.0, []
    for i, v in enumerate(x):
        g = max(0.0, g + v - mu0 - kappa)
        if g > h:
            out.append(i); g = 0.0
    return out

alarms = cusum(x_cp, x_cp[:100].mean())
alarm_t = alarms[0]
x_null = r5.normal(0, 1, 300)
print('first alarm', alarm_t, '| false alarms without shift:', len(cusum(x_null, x_null[:100].mean())))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Anomaly Detection](Anomaly%20Detection.md).*